In [5]:
import os
import certifi

# 1. PostgreSQL 관련 SSL 인증서 오류 해결
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['SSL_CERT_FILE'] = certifi.where()

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime

# 브라우저 설정
options = Options()
# options.add_argument("--headless") # 필요 시 주석 해제
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

url = "https://composecoffee.com/findstore"
driver.get(url)
wait = WebDriverWait(driver, 10)

all_data = []

def parse_with_bs4(html_source):
    """BeautifulSoup으로 현재 페이지의 매장 리스트 파싱 (3번 과정 반영)"""
    soup = BeautifulSoup(html_source, 'html.parser')
    # 2번: item_list 클래스 내의 li들 추출
    items = soup.select(".item_list li") 
    
    page_items = []
    for item in items:
        try:
            # 2번: onclick에서 위도, 경도, 지점ID 추출
            onclick_val = item.get("onclick", "")
            pattern = r"goto_position\(([\d\.]+),\s*([\d\.]+),\s*(\d+)\)"
            match = re.search(pattern, onclick_val)
            lat, lon, store_id = match.groups() if match else (None, None, None)
            
            # 3번: item_right 내부 상세 정보 추출
            right_div = item.select_one(".item_right")
            if not right_div: continue
            
            title = right_div.select_one(".item_title").get_text(strip=True)
            address = right_div.select_one(".item_address").get_text(strip=True)
            phone = right_div.select_one(".item_phone_number").get_text(strip=True)
            
            # 4번 컬럼 순서: 지점ID, 매점명, 주소, 전화번호, 위도, 경도
            page_items.append([store_id, title, address, phone, lat, lon])
        except Exception:
            continue
    return page_items

try:
    # 1번 & 6번: 지역 선택 루프 (gid 0 ~ 7)
    for gid in range(8):
        print(f"\n[지역 전환] gid {gid} 수집 시작...")
        
        # 지역 버튼 클릭
        region_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, f"span.region[gid='{gid}']")))
        region_btn.click()
        time.sleep(2) # 지역 변경 후 로딩 대기

        while True:
            # 현재 페이지 데이터 수집 (BeautifulSoup 사용)
            html = driver.page_source
            page_data = parse_with_bs4(html)
            all_data.extend(page_data)

            # 7번: 페이지 완료 시간 콘솔 출력
            now = datetime.now().strftime('%H:%M:%S')
            
            # 현재 페이지 번호 찾기 (strong 태그)
            soup = BeautifulSoup(html, 'html.parser')
            pagination = soup.select_one(".pagination")
            current_page_num = int(pagination.select_one("strong").get_text(strip=True))
            
            print(f"  > [{now}] gid {gid} - {current_page_num}페이지 수집 완료 (현재 지역 누적: {len(all_data)}개)")

            # 5번 & 6번: 다음 페이지 이동 로직 (숫자가 없을 때까지)
            try:
                # 다음 페이지 번호는 현재번호 + 1
                next_page_num = current_page_num + 1
                
                # pagination 내에서 텍스트가 next_page_num인 a 태그를 찾음
                # 예: <strong>1</strong> 다음에 있는 <a ...>2</a>를 찾음
                next_page_link = driver.find_element(By.XPATH, f"//div[@class='pagination']//a[text()='{next_page_num}']")
                
                if next_page_link:
                    driver.execute_script("arguments[0].click();", next_page_link) # 안전한 클릭
                    time.sleep(1.5) # 페이지 로딩 대기
                else:
                    break # 다음 숫자가 없으면 해당 지역 종료
            except:
                # 다음 숫자 버튼이 없으면(마지막 페이지면) 루프 탈출
                break

        # 7번: 다음 지역으로 넘어갈 때 진행률(%) 표시
        progress = ((gid + 1) / 8) * 100
        print(f"--- [중간 리포트] gid {gid} 완료! 현재 진행도: {progress:.1f}% ---")

    # 4번 & 8번: 데이터 저장
    columns = ["지점ID", "매점명", "주소", "전화번호", "위도", "경도"]
    df = pd.DataFrame(all_data, columns=columns)
    
    # 중복 제거 (지점ID 기준)
    df.drop_duplicates(subset=["지점ID"], keep="first", inplace=True)
    df.to_csv("compose.csv", index=False, encoding="utf-8-sig")
    
    print(f"\n✨ 모든 작업 완료! 총 {len(df)}개의 매장 정보를 'compose.csv'에 저장했습니다.")

except Exception as e:
    print(f"🚨 오류 발생: {e}")

finally:
    driver.quit()


[지역 전환] gid 0 수집 시작...
  > [16:15:20] gid 0 - 1페이지 수집 완료 (현재 지역 누적: 20개)
  > [16:15:22] gid 0 - 2페이지 수집 완료 (현재 지역 누적: 40개)
  > [16:15:25] gid 0 - 3페이지 수집 완료 (현재 지역 누적: 60개)
  > [16:15:28] gid 0 - 4페이지 수집 완료 (현재 지역 누적: 80개)
  > [16:15:30] gid 0 - 5페이지 수집 완료 (현재 지역 누적: 100개)
  > [16:15:32] gid 0 - 6페이지 수집 완료 (현재 지역 누적: 120개)
  > [16:15:35] gid 0 - 7페이지 수집 완료 (현재 지역 누적: 140개)
  > [16:15:37] gid 0 - 8페이지 수집 완료 (현재 지역 누적: 160개)
  > [16:15:39] gid 0 - 9페이지 수집 완료 (현재 지역 누적: 180개)
  > [16:15:41] gid 0 - 10페이지 수집 완료 (현재 지역 누적: 200개)
  > [16:15:43] gid 0 - 11페이지 수집 완료 (현재 지역 누적: 220개)
  > [16:15:45] gid 0 - 12페이지 수집 완료 (현재 지역 누적: 240개)
  > [16:15:48] gid 0 - 13페이지 수집 완료 (현재 지역 누적: 260개)
  > [16:15:50] gid 0 - 14페이지 수집 완료 (현재 지역 누적: 280개)
  > [16:15:52] gid 0 - 15페이지 수집 완료 (현재 지역 누적: 300개)
  > [16:15:54] gid 0 - 16페이지 수집 완료 (현재 지역 누적: 320개)
  > [16:15:56] gid 0 - 17페이지 수집 완료 (현재 지역 누적: 340개)
  > [16:15:59] gid 0 - 18페이지 수집 완료 (현재 지역 누적: 360개)
  > [16:16:01] gid 0 - 19페이지 수집 완료 (현재 지역 누적: 380개)
 

In [2]:
import pandas as pd
import re

# 1. CSV 파일 로드
df = pd.read_csv('compose.csv')

# 2. 한글 포함 여부를 체크하는 함수 정의
def has_hangul(text):
    if pd.isna(text):
        return False
    # 정규표현식으로 한글([가-힣])이 포함되어 있는지 확인
    return bool(re.search('[가-힣]', str(text)))

# 3. '전화번호' 컬럼에서 한글이 포함된 행만 필터링
hangul_phones = df[df['전화번호'].apply(has_hangul)]

# 4. 결과 출력
if not hangul_phones.empty:
    print(f"--- 전화번호 컬럼에 한글이 포함된 데이터 ({len(hangul_phones)}건) ---")
    print(hangul_phones[['지점ID', '매점명', '전화번호', '주소']])
else:
    print("전화번호 컬럼에 한글이 포함된 데이터가 없습니다.")

--- 전화번호 컬럼에 한글이 포함된 데이터 (8건) ---
        지점ID       매점명                       전화번호  \
316   224921     용인초당점                       103호   
477   176197     용인구성점  상가동 212호(마북동, 연원마을 벽산아파트)   
978    65997  원주원동사거리점       강원특별자치도 원주시 원동 278-1   
1468   33460     대구학정점                       102호   
2511    3640      선릉역점            유선전화가 없는 매장입니다.   
2519    3632      신당동점            유선전화가 없는 매장입니다.   
2523    3626    오목교중앙점            유선전화가 없는 매장입니다.   
2570    3479     양천구청점            유선전화가 없는 매장입니다.   

                                         주소  
316                경기 용인시 기흥구 동백중앙로 55 (중동)  
477   경기 용인시 기흥구 용구대로 2398 (마북동, 연원마을벽산아파트)  
978       강원특별자치도 원주시 무실로 113 070-7803-8884  
1468                   대구 북구 학남로3길 40 (학정동)  
2511         서울 강남구 테헤란로63길 12 B동 지상1층 102호  
2519                  서울 중구 다산로 223 1층 103호  
2523         서울 양천구 목동동로12길 45 상가동 1층 11-1호  
2570                 서울 양천구 목동동로 73 1층 102호  


In [1]:
import pandas as pd
import re
from sqlalchemy import create_engine, Integer, String, types
import os
import certifi

# 1. SSL 인증서 오류 방지 (PostgreSQL 충돌 해결)
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()
os.environ['SSL_CERT_FILE'] = certifi.where()

# 2. 데이터 로드 및 정제
df = pd.read_csv('compose.csv')

def clean_phone_number(text):
    """한글이 포함된 전화번호는 결측치(None)로 반환"""
    if pd.isna(text):
        return None
    # 한글([가-힣])이 포함되어 있는지 체크
    if re.search('[가-힣]', str(text)):
        return None
    return text

# '전화번호' 컬럼 정제 적용
df['전화번호'] = df['전화번호'].apply(clean_phone_number)

# 3. MySQL 연결 설정 (본인의 환경에 맞게 수정)
user = "root"          
password = "1234"  
host = "localhost"     
port = "3306"          
db_name = "coffee_store" 

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:{port}/{db_name}?charset=utf8mb4")

# 4. SQLAlchemy를 사용한 데이터 삽입
try:
    # 결측치(NaN)를 DB의 NULL로 인식하도록 처리
    df_to_db = df.where(pd.notnull(df), None)

    df_to_db.to_sql(
        name='compose',      
        con=engine,          
        if_exists='replace', # 기존 테이블 삭제 후 재생성
        index=False,         
        dtype={
            '지점ID': Integer(),      
            '매점명': String(100),    
            '주소': String(550),      
            '전화번호': String(20),     # 정제되었으므로 50자면 충분합니다.
            '위도': types.DECIMAL(10, 7), 
            '경도': types.DECIMAL(10, 7)  
        }
    )
    print(f"✅ 정제 및 저장 완료! 총 {len(df)}개의 매장 데이터가 저장되었습니다.")
    
    # 정제된 내역 확인용 출력
    print("\n--- 정제 결과 예시 (한글 포함되었던 데이터들) ---")
    hangul_check_list = [224921, 176197, 65997, 3640] # 이전에 확인된 지점ID들
    print(df[df['지점ID'].isin(hangul_check_list)][['매점명', '전화번호']])

except Exception as e:
    print(f"🚨 오류 발생: {e}")

✅ 정제 및 저장 완료! 총 2926개의 매장 데이터가 저장되었습니다.

--- 정제 결과 예시 (한글 포함되었던 데이터들) ---
           매점명  전화번호
316      용인초당점  None
477      용인구성점  None
978   원주원동사거리점  None
2511      선릉역점  None
